# Infrastructure Risk Metrics

This notebook creates sub-national infrastructure risk metrics for the national tool.

It is designed to grow section by section. Each hazard section creates a tidy metrics table keyed by `adm_id`; the final section exports one CSV per hazard plus a combined hazard-indexed CSV.

## 0. Setup

Set the country, administrative level, hazard runs, and input/output paths here. The notebook can summarize multiple hazards in one run while keeping `hazard` and `model_run` explicit in the output.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import geopandas as gpd

pd.set_option("display.max_columns", 100)

In [2]:
# Core parameters
COUNTRY_ISO3 = "KEN"
COUNTRY_NAME = "Kenya"
ADMIN_LEVEL = "adm1"  # options currently available: adm0, adm1, adm2
MODULE = "infrastructure"

# Resolve paths whether the notebook is run from the repo root or notebooks/.
NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = NOTEBOOK_CWD.parent if NOTEBOOK_CWD.name == "notebooks" else NOTEBOOK_CWD

BOUNDARY_PATH = REPO_ROOT / "data" / "boundaries" / COUNTRY_ISO3 / ADMIN_LEVEL / f"{COUNTRY_ISO3}_{ADMIN_LEVEL}.shp"
RAW_INFRASTRUCTURE_DIR = REPO_ROOT / "data" / "raw" / COUNTRY_ISO3 / MODULE
OUTPUT_ROOT = REPO_ROOT / "results" / COUNTRY_ISO3 / MODULE
COMBINED_OUTPUT_PATH = OUTPUT_ROOT / f"{COUNTRY_ISO3}_{ADMIN_LEVEL}_{MODULE}_metrics_by_hazard.csv"

# Flooding has more than one available source/scenario. Select the run that should feed the reported flooding metrics.
SELECTED_FLOODING_RUN = "river-jrc_floodMapGL"
FLOODING_RUNS = {
    "river-jrc_floodMapGL": {
        "model_run": "river-jrc_floodMapGL",
        "road_path": RAW_INFRASTRUCTURE_DIR / "flooding" / "river-jrc_road_damages.gpkg",
        "rail_path": RAW_INFRASTRUCTURE_DIR / "flooding" / "river-jrc_rail_damages.gpkg",
        "road_ead_column": "hazard-floodMapGL_EAD",
        "rail_ead_column": "hazard-floodMapGL_EAD",
    },
    "river-aqueduct_historical_mean_1980": {
        "model_run": "river-aqueduct_historical_mean_1980",
        "road_path": RAW_INFRASTRUCTURE_DIR / "flooding" / "river-aqueduct_road_damages.gpkg",
        "rail_path": RAW_INFRASTRUCTURE_DIR / "flooding" / "river-aqueduct_rail_damages.gpkg",
        "road_ead_column": "hazard-inunriver_historical_MEAN_1980_EAD",
        "rail_ead_column": "hazard-inunriver_historical_MEAN_1980_EAD",
    },
}

# TC power data currently come from the STORM tropical cyclone model.
SELECTED_TC_RUN = "storm_baseline_2020"
TC_RUNS = {
    "storm_baseline_2020": {
        "model_run": "storm_baseline_2020",
        "power_path": RAW_INFRASTRUCTURE_DIR / "tc" / "power.gpkg",
        "power_ead_column": "ead__cyclone__rcp_baseline__epoch_2020",
    },
}

BOUNDARY_PATH, RAW_INFRASTRUCTURE_DIR, COMBINED_OUTPUT_PATH

(WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/data/boundaries/KEN/adm1/KEN_adm1.shp'),
 WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/data/raw/KEN/infrastructure'),
 WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/results/KEN/infrastructure/KEN_adm1_infrastructure_metrics_by_hazard.csv'))

## 0.1 Load Administrative Boundaries

The current Kenya boundary files use `shapeID` as the stable administrative identifier and `shapeName` as the display name.

In [3]:
ADMIN_ID_FIELD = "shapeID"
ADMIN_NAME_FIELD = "shapeName"

boundaries = gpd.read_file(BOUNDARY_PATH)

required_boundary_cols = {ADMIN_ID_FIELD, ADMIN_NAME_FIELD, "geometry"}
missing_boundary_cols = required_boundary_cols.difference(boundaries.columns)
if missing_boundary_cols:
    raise ValueError(f"Boundary layer is missing required columns: {sorted(missing_boundary_cols)}")

admin_regions = boundaries[[ADMIN_ID_FIELD, ADMIN_NAME_FIELD, "geometry"]].copy()
admin_regions = admin_regions.rename(columns={ADMIN_ID_FIELD: "adm_id", ADMIN_NAME_FIELD: "adm_name"})
admin_regions["country_iso3"] = COUNTRY_ISO3
admin_regions["country_name"] = COUNTRY_NAME
admin_regions["admin_level"] = ADMIN_LEVEL.upper()
admin_regions["module"] = MODULE

admin_regions.head()

,adm_id,adm_name,geometry,country_iso3,country_name,admin_level,module
0,32016919B72266624462344,Turkana,"POLYGON ((35.94724 4.63093, 35.94857 4.62942, ...",KEN,Kenya,ADM1,infrastructure
1,32016919B63496705134089,Marsabit,"POLYGON ((36.71131 4.44592, 36.72669 4.44664, ...",KEN,Kenya,ADM1,infrastructure
2,32016919B2031803566233,Mandera,"POLYGON ((39.77523 3.6681, 39.86773 3.8675, 39...",KEN,Kenya,ADM1,infrastructure
3,32016919B89873713911655,Wajir,"POLYGON ((39.31809 3.47197, 39.33258 3.46684, ...",KEN,Kenya,ADM1,infrastructure
4,32016919B96045830258165,West Pokot,"POLYGON ((34.93152 2.43647, 34.93121 2.43842, ...",KEN,Kenya,ADM1,infrastructure


## 0.2 Helper Functions

Damage records are line assets with EAD values. When a line crosses administrative boundaries, the notebook intersects the line with each region and allocates EAD in proportion to intersected line length.

In [4]:
def check_path_exists(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


def normalize_metric_token(value: object) -> str:
    '''Convert a category value into a compact metric-name token.'''
    token = str(value).lower().strip()
    token = re.sub(r"^(road|rail|power)_", "", token)
    token = re.sub(r"[^a-z0-9]+", "_", token).strip("_")
    return token or "unknown"


def validate_columns(frame: pd.DataFrame, required_columns: set[str], label: str) -> None:
    missing_columns = required_columns.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{label} is missing required columns: {sorted(missing_columns)}")


def line_ead_by_admin(
    asset_path: Path,
    polygons: gpd.GeoDataFrame,
    ead_column: str,
    total_metric_name: str,
    group_column: str | None = None,
    group_metric_prefix: str | None = None,
) -> pd.DataFrame:
    '''Apportion line-asset EAD to admin regions by intersected line length.'''
    check_path_exists(asset_path, "Asset damage GeoPackage")

    assets = gpd.read_file(asset_path)
    required_columns = {ead_column, "geometry"}
    if group_column:
        required_columns.add(group_column)
    validate_columns(assets, required_columns, asset_path.name)

    assets = assets[~assets.geometry.is_empty & assets.geometry.notna()].copy()
    if assets.empty:
        raise ValueError(f"No valid line geometries found in {asset_path}")

    assets[ead_column] = pd.to_numeric(assets[ead_column], errors="coerce").fillna(0)
    if group_column:
        assets[group_column] = assets[group_column].fillna("unknown")

    metric_crs = polygons.estimate_utm_crs() or "EPSG:3857"
    admin_for_overlay = polygons[["adm_id", "geometry"]].to_crs(metric_crs)
    assets_for_overlay = assets.to_crs(metric_crs).copy()
    assets_for_overlay["_asset_row_id"] = np.arange(len(assets_for_overlay))
    assets_for_overlay["_asset_length_m"] = assets_for_overlay.geometry.length
    assets_for_overlay = assets_for_overlay[assets_for_overlay["_asset_length_m"] > 0].copy()

    keep_columns = ["_asset_row_id", "_asset_length_m", ead_column, "geometry"]
    if group_column:
        keep_columns.insert(3, group_column)

    intersections = gpd.overlay(
        assets_for_overlay[keep_columns],
        admin_for_overlay,
        how="intersection",
        keep_geom_type=True,
    )

    result = polygons[["adm_id"]].copy()
    if intersections.empty:
        result[total_metric_name] = 0.0
        return result

    intersections = intersections[~intersections.geometry.is_empty & intersections.geometry.notna()].copy()
    intersections["_intersection_length_m"] = intersections.geometry.length
    intersections["_length_weight"] = intersections["_intersection_length_m"] / intersections["_asset_length_m"]
    intersections["_ead_apportioned"] = intersections[ead_column] * intersections["_length_weight"]

    total_by_admin = intersections.groupby("adm_id")["_ead_apportioned"].sum().rename(total_metric_name)
    result = result.merge(total_by_admin.reset_index(), on="adm_id", how="left", validate="one_to_one")
    result[total_metric_name] = result[total_metric_name].fillna(0)

    if group_column:
        if group_metric_prefix is None:
            group_metric_prefix = total_metric_name
        grouped = intersections.groupby(["adm_id", group_column])["_ead_apportioned"].sum().reset_index()
        grouped["metric_name"] = grouped[group_column].map(
            lambda value: f"{group_metric_prefix}_{normalize_metric_token(value)}"
        )
        by_group = grouped.pivot_table(index="adm_id", columns="metric_name", values="_ead_apportioned", fill_value=0)
        by_group.columns.name = None
        result = result.merge(by_group.reset_index(), on="adm_id", how="left", validate="one_to_one")
        group_metric_columns = [column for column in result.columns if column.startswith(f"{group_metric_prefix}_")]
        result[group_metric_columns] = result[group_metric_columns].fillna(0)

    return result


def build_hazard_metrics(
    metric_tables: list[pd.DataFrame],
    hazard: str,
    model_run: str,
) -> pd.DataFrame:
    '''Combine a hazard's section tables and attach standard identifiers.'''
    identifier_columns = ["country_iso3", "country_name", "admin_level", "adm_id", "adm_name", "module"]
    hazard_metrics = admin_regions[identifier_columns].copy()
    hazard_metrics["hazard"] = hazard
    hazard_metrics["model_run"] = model_run

    for metrics in metric_tables:
        if "adm_id" not in metrics.columns:
            raise ValueError("Each section metrics table must include an 'adm_id' column")
        hazard_metrics = hazard_metrics.merge(metrics, on="adm_id", how="left", validate="one_to_one")

    numeric_columns = hazard_metrics.select_dtypes(include="number").columns
    hazard_metrics[numeric_columns] = hazard_metrics[numeric_columns].fillna(0).round(3)
    return hazard_metrics

## 0.3 Hazard Reporting Strategy

Keep `hazard` as a reporting dimension, and keep `model_run` as the selected source/scenario within that hazard, including the model name where it is known. This lets the tool filter across hazards while avoiding ambiguous metric names when multiple flood or cyclone products are available.

The notebook exports both hazard-specific files and a combined table. Hazard-specific files are convenient for module pipelines; the combined table is convenient for dashboard filters.

## 1. Flooding EAD

This section summarizes flooding expected annual damages for road and rail assets.

Metrics being reported:

- `road_ead_total`: total road EAD in each administrative region.
- `road_ead_<asset_type>`: road EAD by road asset type.
- `rail_ead_total`: total rail EAD in each administrative region.

In [5]:
flooding_run = FLOODING_RUNS[SELECTED_FLOODING_RUN]

for path_label in ["road_path", "rail_path"]:
    check_path_exists(flooding_run[path_label], path_label)

flooding_run

{'model_run': 'river-jrc_floodMapGL',
 'road_path': WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/data/raw/KEN/infrastructure/flooding/river-jrc_road_damages.gpkg'),
 'rail_path': WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/data/raw/KEN/infrastructure/flooding/river-jrc_rail_damages.gpkg'),
 'road_ead_column': 'hazard-floodMapGL_EAD',
 'rail_ead_column': 'hazard-floodMapGL_EAD'}

In [6]:
road_ead_metrics = line_ead_by_admin(
    flooding_run["road_path"],
    admin_regions,
    ead_column=flooding_run["road_ead_column"],
    total_metric_name="road_ead_total",
    group_column="asset_type",
    group_metric_prefix="road_ead",
)

rail_ead_metrics = line_ead_by_admin(
    flooding_run["rail_path"],
    admin_regions,
    ead_column=flooding_run["rail_ead_column"],
    total_metric_name="rail_ead_total",
)

# Future flooding sections should append their metrics DataFrames to this list.
flooding_section_metric_tables = [
    road_ead_metrics,
    rail_ead_metrics,
]

flooding_metrics = build_hazard_metrics(
    flooding_section_metric_tables,
    hazard="flooding",
    model_run=flooding_run["model_run"],
)

flooding_metrics.head()

,country_iso3,country_name,admin_level,adm_id,adm_name,module,hazard,model_run,road_ead_total,road_ead_motorway,road_ead_primary,road_ead_secondary,road_ead_tertiary,road_ead_trunk,rail_ead_total
0,KEN,Kenya,ADM1,32016919B72266624462344,Turkana,infrastructure,flooding,river-jrc_floodMapGL,167056.972,0.0,0.0,29620.444,126075.285,11361.243,0.0
1,KEN,Kenya,ADM1,32016919B63496705134089,Marsabit,infrastructure,flooding,river-jrc_floodMapGL,129494.079,0.0,0.0,92317.777,13689.585,23486.716,0.0
2,KEN,Kenya,ADM1,32016919B2031803566233,Mandera,infrastructure,flooding,river-jrc_floodMapGL,175472.091,0.0,0.0,112008.942,57655.927,5807.222,0.0
3,KEN,Kenya,ADM1,32016919B89873713911655,Wajir,infrastructure,flooding,river-jrc_floodMapGL,973902.545,0.0,0.0,91668.619,851568.809,30665.117,0.0
4,KEN,Kenya,ADM1,32016919B96045830258165,West Pokot,infrastructure,flooding,river-jrc_floodMapGL,40276.847,0.0,0.0,12878.939,27397.908,0.000,0.0


## 2. Tropical Cyclone EAD

This section summarizes baseline tropical cyclone expected annual damages for the power network. The current power-network dataset is from the STORM tropical cyclone model.

Metrics being reported:

- `power_ead_baseline`: total baseline power-network EAD in each administrative region.

In [7]:
tc_run = TC_RUNS[SELECTED_TC_RUN]
check_path_exists(tc_run["power_path"], "Power damage GeoPackage")

tc_run

{'model_run': 'storm_baseline_2020',
 'power_path': WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/data/raw/KEN/infrastructure/tc/power.gpkg'),
 'power_ead_column': 'ead__cyclone__rcp_baseline__epoch_2020'}

In [8]:
power_ead_metrics = line_ead_by_admin(
    tc_run["power_path"],
    admin_regions,
    ead_column=tc_run["power_ead_column"],
    total_metric_name="power_ead_baseline",
)

# Future TC sections, such as indirect risk from power disruption, should append their metrics DataFrames here.
tc_section_metric_tables = [
    power_ead_metrics,
]

tc_metrics = build_hazard_metrics(
    tc_section_metric_tables,
    hazard="tc",
    model_run=tc_run["model_run"],
)

tc_metrics.head()

,country_iso3,country_name,admin_level,adm_id,adm_name,module,hazard,model_run,power_ead_baseline
0,KEN,Kenya,ADM1,32016919B72266624462344,Turkana,infrastructure,tc,storm_baseline_2020,0.0
1,KEN,Kenya,ADM1,32016919B63496705134089,Marsabit,infrastructure,tc,storm_baseline_2020,0.0
2,KEN,Kenya,ADM1,32016919B2031803566233,Mandera,infrastructure,tc,storm_baseline_2020,0.0
3,KEN,Kenya,ADM1,32016919B89873713911655,Wajir,infrastructure,tc,storm_baseline_2020,0.0
4,KEN,Kenya,ADM1,32016919B96045830258165,West Pokot,infrastructure,tc,storm_baseline_2020,0.0


## 3. Combine And Export Infrastructure Metrics

This final section writes one CSV per hazard and one combined hazard-indexed CSV.

In [9]:
hazard_metric_outputs = {
    "flooding": flooding_metrics,
    "tc": tc_metrics,
}

for hazard, metrics in hazard_metric_outputs.items():
    hazard_output_dir = OUTPUT_ROOT / hazard
    hazard_output_dir.mkdir(parents=True, exist_ok=True)
    hazard_output_path = hazard_output_dir / f"{COUNTRY_ISO3}_{ADMIN_LEVEL}_{MODULE}_{hazard}_metrics.csv"
    metrics.to_csv(hazard_output_path, index=False)
    print(f"Exported {len(metrics):,} {hazard} rows and {len(metrics.columns):,} columns to:")
    print(hazard_output_path)

combined_metrics = pd.concat(hazard_metric_outputs.values(), ignore_index=True, sort=False)
combined_metrics.to_csv(COMBINED_OUTPUT_PATH, index=False)

print(f"Exported {len(combined_metrics):,} combined rows and {len(combined_metrics.columns):,} columns to:")
print(COMBINED_OUTPUT_PATH)

Exported 47 flooding rows and 15 columns to:
C:\Users\Mark.DESKTOP-UFHIN6T\Projects\national_tool_metrics\results\KEN\infrastructure\flooding\KEN_adm1_infrastructure_flooding_metrics.csv
Exported 47 tc rows and 9 columns to:
C:\Users\Mark.DESKTOP-UFHIN6T\Projects\national_tool_metrics\results\KEN\infrastructure\tc\KEN_adm1_infrastructure_tc_metrics.csv
Exported 94 combined rows and 16 columns to:
C:\Users\Mark.DESKTOP-UFHIN6T\Projects\national_tool_metrics\results\KEN\infrastructure\KEN_adm1_infrastructure_metrics_by_hazard.csv
